# Modelo XGBoost

## 1. Librerias

In [ ]:
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import s3fs

from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from matplotlib import LinearSegmentedColormap

## 2. Configuración global

In [ ]:

S3_BASE = "s3://tfm-mu-bd/processed"
S3_OUT = "s3://tfm-mu-bd/outputs/xgboost_complete"
s3 = s3fs.S3FileSystem()

EXCLUDE_COLS = [
    "record_id", "dx", "labels", "categories",
    "is_adverse", "snomed_unknown", "dx_multi_hot", "cat_arrhythmia",
    "cat_axis_deviation", "cat_conduction_block", "cat_interval_change", "cat_ischemia",        
    "cat_morphology_change", "cat_repolarization",  
    "cat_sinus_rhythm", "cat_structural"   
]

RANDOM_STATE = 42
MIN_PRECISION = 0.20  

colors = [
    "#cdeae5",
    "#a8dadc",
    "#77abbd",
    "#457b9d",
    "#31587a",
    "#1d3557",
    "#324766"
]
cmap = LinearSegmentedColormap.from_list("custom_blue", colors)
C_MID = "#457b9d"
C_DARK = "#1d3557"
C_ACCENT = "#77abbd"
C_BG = "#f4f9fb"
C_GRID = "#dce9ef"

## 3. Carga y preparación de datos

In [ ]:
def prepare_features(df, feature_cols):
    X = df[feature_cols].copy()
    for col in feature_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    return X.values.astype(np.float32)


def load_and_prepare():
    print("Cargando datos desde S3")

    df_train = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_train.parquet")
    df_val = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_val.parquet")
    df_test = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_test.parquet")

    feature_cols = [c for c in df_train.columns if c not in EXCLUDE_COLS]

    imputer = SimpleImputer(strategy="median")

    X_train = imputer.fit_transform(prepare_features(df_train, feature_cols))
    X_val = imputer.transform(prepare_features(df_val, feature_cols))
    X_test = imputer.transform(prepare_features(df_test, feature_cols))

    y_train = df_train["is_adverse"].astype(int).values
    y_val = df_val["is_adverse"].astype(int).values
    y_test = df_test["is_adverse"].astype(int).values

    classes = np.array([0, 1])
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    scale_pos_weight = weights[1] / weights[0]

    print(f"scale_pos_weight: {scale_pos_weight:.4f}")
    print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols, imputer, scale_pos_weight

## 4. Tuning y entrenamiento

In [ ]:
def tune_xgboost(X_train, y_train, scale_pos_weight):

    print("\nTUNING DE HIPERPARAMETROS")

    param_dist = {
        "n_estimators": [200, 300, 500],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "min_child_weight": [1, 3, 5],
        "gamma": [0, 0.1],
        "reg_alpha": [0, 0.1],
        "reg_lambda": [1, 1.5],
    }

    base_model = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    search = RandomizedSearchCV(
        base_model,
        param_distributions=param_dist,
        n_iter=20, 
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    print("Best params:", search.best_params_)
    print("Best CV:", search.best_score_)

    return search.best_estimator_, search.best_params_, search.best_score_

def train_final(best_params, X_train, y_train, X_val, y_val, scale_pos_weight):

    print("\n ENTRENAMIENTO FINAL")

    model = XGBClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )

    print("Best iteration:", model.best_iteration)

    return model


def find_best_threshold(model, X_val, y_val, min_recall=0.80):

    y_prob = model.predict_proba(X_val)[:, 1]
    thresholds = np.linspace(0.1, 0.9, 100)

    best_t = 0.5
    best_f1 = 0

    for t in thresholds:
        preds = (y_prob > t).astype(int)
        rec = recall_score(y_val, preds)
        f1  = f1_score(y_val, preds)

        if rec >= min_recall and f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"\nBest threshold: {best_t:.3f} | F1: {best_f1:.4f}")
    return best_t


## 6. Evaluación

In [ ]:
def evaluate_full(model, X, y, split_name, threshold):

    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob > threshold).astype(int)

    auc = roc_auc_score(y, y_prob)
    f1 = f1_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    ap = average_precision_score(y, y_prob)

    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)

    print(f"\n[{split_name}]")
    print("AUC:", auc)
    print("F1:", f1)
    print("Recall:", rec)
    print("Threshold:", threshold)
    print(cm)

    return {
        "split": split_name,
        "auc_roc": round(auc, 4),
        "auc_pr": round(ap, 4),
        "f1": round(f1, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "specificity": round(specificity, 4),
        "threshold": threshold,
        "cm": cm.tolist(),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
    }


## 7. Visualización

In [ ]:
def _apply_style(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor(C_BG)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_edgecolor(C_GRID)
    if title: ax.set_title(title, fontsize=12, fontweight="bold", color=C_DARK, pad=10)
    if xlabel: ax.set_xlabel(xlabel, fontsize=10, color=C_DARK)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10, color=C_DARK)
    ax.tick_params(colors=C_DARK, labelsize=9)


def plot_results(model, X_val, y_val, X_test, y_test, feature_cols, threshold, top_n=20, save_path=None):

    def _save(fig, suffix):
        if save_path:
            fig.savefig(save_path.replace(".", f"_{suffix}."), dpi=150, bbox_inches="tight", facecolor="white")
        plt.show()

    # Probabilidades y predicciones
    y_prob_val = model.predict_proba(X_val)[:, 1]
    y_prob_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = (y_prob_test > threshold).astype(int)

    # Curvas ROC
    fpr_v, tpr_v, _ = roc_curve(y_val,  y_prob_val)
    fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_test)
    auc_v = roc_auc_score(y_val,  y_prob_val)
    auc_t = roc_auc_score(y_test, y_prob_test)

    fig1, ax = plt.subplots(figsize=(7, 6), facecolor="white")
    ax.plot(fpr_v, tpr_v, color=C_MID, lw=2.2, label=f"Val  (AUC = {auc_v:.3f})")
    ax.plot(fpr_t, tpr_t, color=C_ACCENT, lw=2.2, label=f"Test (AUC = {auc_t:.3f})", linestyle="--")
    ax.plot([0, 1], [0, 1], color=C_DARK, lw=1, linestyle=":", alpha=0.5, label="Random")
    ax.set(xlim=[-0.02, 1.02], ylim=[-0.02, 1.05])
    _apply_style(ax, "AUC-ROC", "False Positive Rate", "True Positive Rate")
    ax.legend(fontsize=9, framealpha=0.9, edgecolor=C_GRID)
    plt.tight_layout()
    _save(fig1, "roc")

    # Curvas PR
    prec_v, rec_v, _ = precision_recall_curve(y_val,  y_prob_val)
    prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_test)
    ap_v = average_precision_score(y_val,  y_prob_val)
    ap_t = average_precision_score(y_test, y_prob_test)

    fig2, ax = plt.subplots(figsize=(7, 6), facecolor="white")
    ax.plot(rec_v, prec_v, color=C_MID, lw=2.2, label=f"Val  (AP = {ap_v:.3f})")
    ax.plot(rec_t, prec_t, color=C_ACCENT, lw=2.2, label=f"Test (AP = {ap_t:.3f})", linestyle="--")
    ax.axhline(y_test.mean(), color=C_DARK, lw=1, linestyle=":", alpha=0.5, label=f"Baseline ({y_test.mean():.2f})")
    ax.set(xlim=[-0.02, 1.02], ylim=[-0.02, 1.05])
    _apply_style(ax, "AUC-PR (Precision-Recall)", "Recall", "Precision")
    ax.legend(fontsize=9, framealpha=0.9, edgecolor=C_GRID)
    plt.tight_layout()
    _save(fig2, "pr")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_test)
    fig3, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, cbar=False,
                linewidths=1, linecolor="white",
                xticklabels=["No adverso", "Adverso"],
                yticklabels=["No adverso", "Adverso"], ax=ax)
    ax.set_title(f"Confusion Matrix — Test\n(threshold = {threshold:.3f})",
                 fontsize=12, fontweight="bold", color=C_DARK, pad=10)
    ax.set(xlabel="Predicción", ylabel="Real")
    ax.tick_params(colors=C_DARK, labelsize=9)
    plt.tight_layout()
    _save(fig3, "cm")

    # Feature Importance
    fi_df = (pd.DataFrame({"feature": feature_cols, "importance": model.feature_importances_})
               .sort_values("importance", ascending=True)
               .tail(top_n))
    fig4, ax = plt.subplots(figsize=(8, top_n * 0.4 + 1), facecolor="white")
    bar_colors = [cmap(v) for v in np.linspace(0.25, 0.95, len(fi_df))]
    bars = ax.barh(fi_df["feature"], fi_df["importance"],
                   color=bar_colors, edgecolor="white", linewidth=0.5, height=0.7)
    for bar, val in zip(bars, fi_df["importance"]):
        ax.text(bar.get_width() + fi_df["importance"].max() * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=7.5, color=C_DARK)
    _apply_style(ax, f"Feature Importance (Top {top_n})", "Importance (gain)", "")
    ax.tick_params(axis="y", labelsize=8)
    ax.margins(x=0.15)
    plt.tight_layout()
    _save(fig4, "fi")

    return fig1, fig2, fig3, fig4

## 8. Ejecución principal

In [ ]:
if __name__ == "__main__":

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols, imputer, scale_pos_weight = load_and_prepare()
    best_model, best_params, best_cv = tune_xgboost(X_train, y_train, scale_pos_weight)
    final_model = train_final(best_params, X_train, y_train, X_val, y_val, scale_pos_weight)

    BEST_THRESHOLD = find_best_threshold(final_model, X_val, y_val, min_recall=0.80)
    print("\nEVALUATION")

    r_val  = evaluate_full(final_model, X_val,  y_val,  "VAL",  BEST_THRESHOLD)
    r_test = evaluate_full(final_model, X_test, y_test, "TEST", BEST_THRESHOLD)

    plot_results(
        model = final_model,
        X_val = X_val,
        y_val = y_val,
        X_test = X_test,
        y_test = y_test,
        feature_cols = feature_cols,
        threshold = BEST_THRESHOLD,
        top_n = 20,
        save_path = "xgboost_results.png"
    )

    print("\nDONE")

## 9. Guardar el modelo

In [ ]:
final_model.save_model("/tmp/xgboost_model.json")

s3_client.upload_file(
    "/tmp/xgboost_model.json",
    BUCKET,
    "models/xgboost_model.json"
)